In [ ]:
from data.dataset import RailClsDataset, raildb_row_anchor
import torchvision.transforms as transforms
import data.mytransforms as mytransforms

target_transform = transforms.Compose([
        mytransforms.FreeScaleMask((288, 800)),
        mytransforms.MaskToTensor(),
    ])

img_transform = transforms.Compose([
    transforms.Resize((288, 800)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
simu_transform = None
data_root = "dataset/RailDB/"
griding_num=56
num_rails=4
type = "all"
train_dataset = RailClsDataset(
                data_root,
                data_root+'meta.csv',
                img_transform = img_transform, 
                target_transform = target_transform, 
                simu_transform = simu_transform, 
                griding_num = griding_num, 
                row_anchor = raildb_row_anchor, 
                num_rails = num_rails,
                mode = "train",
                type = type,
                )
val_dataset = RailClsDataset(
                data_root,
                data_root+'meta.csv',
                img_transform = img_transform, 
                target_transform = target_transform, 
                simu_transform = simu_transform, 
                griding_num = griding_num, 
                row_anchor = raildb_row_anchor, 
                num_rails = num_rails,
                mode = "val",
                type = type,
                )

In [30]:
img, grid_label, inter_label, seg_label, jpeg_name = val_dataset.__getitem__(0)


In [ ]:
# view image tensor with inter_label
# denormalize image tensor
import torch 
import matplotlib.pyplot as plt

def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

img_vis = denormalize(img, (0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
img_vis = img_vis.permute(1, 2, 0).numpy()
print(img_vis.shape, jpeg_name)


In [ ]:
import json
import numpy as np

def resize_points(points, original_size, target_size):
    """
    Resize points from original size to target size.
    """
    scale_x = target_size[1] / original_size[1]
    scale_y = target_size[0] / original_size[0]
    return points * np.array([scale_x, scale_y]).reshape(1, 2)

label_file = "dataset/RailDB/anno/fixed_rain/hiv00223_002054.json"
with open(label_file, "r") as file:
    label = json.load(file)

shapes = label['shapes']
print(shapes)

points_list = []
for line in shapes:
    points = np.array(line["points"])
    points = resize_points(points, (720, 1280), img_vis.shape[:2])
    points_list.append(points)
    print(points.shape)
    print(line["points"])

# plt.imshow(img_vis)
# for points in points_list:
#     plt.plot(points[:, 0], points[:, 1])


